# GRM shard batch submission (Google Batch / dsub)

Moves GRM shard construction off the interactive VM and onto Google Batch,
via `dsub` -- one task per shard, each on its own machine. Three stages,
each meant to be run in order, and each a checkpoint before spending more:

1. **Troubleshooting job** -- the smallest possible `dsub` submission
   (no plink, no real inputs), just to confirm submission itself works
   (auth, project/region config, logging bucket permissions, the
   `google-batch` provider) before anything plink-specific can go wrong
   and get confused with dsub-specific problems.
2. **Validation batch** -- a handful of real shards (5, spread across a
   candidate `N_SHARDS`) with the real plink command, to get real
   per-task timing/cost before committing to the full run.
3. **Full run** -- not built yet. Comes after (2) confirms timing/cost are
   sane, and after the fixed-vs-variable-cost question from
   `grm_shard_timing.ipynb` is resolved (does per-shard time stay roughly
   flat going from 2000 shards down to ~300? if so, fewer/larger shards is
   strictly better -- see that notebook's "Next steps").

**Everything here is a first draft.** `dsub` provider names and exact
machine-type string formatting are version-sensitive -- expect to debug
the troubleshooting job itself before it succeeds.

## Prerequisites

`dsub` submits to Google Batch from wherever this cell runs (this
notebook's VM), but the actual work happens on separate Batch-provisioned
machines -- neither the interactive VM's compute resource nor its plink
install matter for what follows. Needs `dsub` installed and `gcloud`
already authenticated to the right project (both should already be true on
a Verily Workbench VM, but confirmed explicitly below rather than assumed).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Fill in `PROJECT_ID`/`REGION` from the `gcloud config` output above if they
didn't print automatically. `MEMORY_MB` is a placeholder -- replace with
the real `max_rss_gb` measured in `grm_shard_timing.ipynb` (with
`--read-freq`/`--memory` applied) plus ~20-30% headroom before running the
validation batch for real; the troubleshooting job below doesn't use it.

In [ ]:
import os
import math

PROJECT_ID = "wb-fulgent-almond-9474"
REGION = "us-central1"   # TODO -- confirm against gcloud config output above

# dsub's default (project's Compute Engine default SA) failed with PERMISSION_DENIED --
# the calling identity doesn't have actAs rights on it. Using this VM's own attached
# "pet" SA instead, which the caller can already act as without an extra IAM grant.
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-fulgent-almond-9474.iam.gserviceaccount.com"

# VPC Service Controls is enabled on this project -- Batch jobs must run with no
# external IP, on the project's private network/subnet. Per Verily Workbench's own
# dsub guide (support.workbench.verily.com/docs/guides/workflows/dsub/), the network
# and subnetwork are literally named "network"/"subnetwork" in every Workbench project.
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
# confirmed via `ps -eo pid,args | grep gcsfuse` -- this local mount's gcsfuse process
# is `gcsfuse ... shared-env-pilot-wb-fulgent-almond-9474 <this path>`, i.e. the bucket
# ROOT maps directly to this mount point, no extra path suffix. The earlier guess
# ("gs://wb-fulgent-almond-9474/shared-env-pilot") used the project ID as the bucket
# name, which doesn't exist (404) -- that's what caused the troubleshooting job's log
# upload, and therefore the job itself, to fail.
WORKSPACE_BUCKET_GS = "gs://shared-env-pilot-wb-fulgent-almond-9474"

BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/data/03_grm_shards"

GRM_INPUT_DIR = f"{BUCKET_DIR}/grm_input"
GRM_INPUT_DIR_GS = f"{BUCKET_DIR_GS}/grm_input"

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = f"{BUCKET_DIR}/{BED_NAME}"
BED_PREFIX_GS = f"{BUCKET_DIR_GS}/{BED_NAME}"   # gs:// form -- needed for a server-side
                                                 # (bucket-to-bucket) copy, see the staging cell

# grm_shard_run.ipynb's local single-shard timing (isolated, no contention) found
# --make-grm-bin is NOT single-threaded (contrary to plink's own docs), and more
# threads keeps helping well past the ~4-6 originally guessed: --threads 12 finished
# in 958.6s vs --threads 4's 1473.3s (54% slower) at the same N_SHARDS. Parallel
# efficiency does drop as threads increase (65% at 4 threads vs 34% at 12), so total
# core-time -- the cost driver -- rises with more vCPUs even as wall-clock falls. Each
# Batch task gets a dedicated VM (no contention from sibling shards), and the dollar
# amounts here are small regardless (~$0.30-0.40/shard either way), so optimizing for
# wall-clock over cost: using the fastest measured setting.
MACHINE_VCPUS = 12

# sized the same way as grm_shard_run.ipynb: 2x bed file size + 4 GB headroom, not a
# guess -- update BED_SIZE_GB below if the bed file changes. N1 custom machine types
# require memory to be an EXACT multiple of 256 MB -- the raw formula's output
# (103628) isn't one, which Batch rejected outright ("Instance properties must
# provide existing machine type", CODE_GCE_BAD_REQUEST) before any task could even
# start. Round up to the nearest 256 MB to guarantee a valid value.
BED_SIZE_GB = 48.6
_raw_memory_mb = (BED_SIZE_GB * 2 + 4) * 1024
MEMORY_MB = math.ceil(_raw_memory_mb / 256) * 256

print(BUCKET_DIR_GS)
print(f"MACHINE_VCPUS={MACHINE_VCPUS}, MEMORY_MB={MEMORY_MB}")

## Stage a clean input folder (one-time)

Each `dsub` task downloads whatever's in this folder -- keep it to just the
4 files plink actually needs, not the whole `03_grm_shards/` bucket
contents (which also has the per-chromosome intermediates, logs, etc.).

In [ ]:
%%bash -s "$BED_PREFIX_GS" "$GRM_INPUT_DIR_GS"
set -e
BED_PREFIX_GS=$1
GRM_INPUT_DIR_GS=$2

# server-side (bucket-to-bucket) copies via gs:// URIs -- NOT `cp` between two
# gcsfuse-mounted paths. gcsfuse has no server-side-copy concept: a plain `cp` between
# two mounted paths reads the whole object down to this VM's local disk and re-uploads
# it, even when source and dest are in the same bucket. For the ~49 GB bed file that's
# both slow and can exhaust local disk ("No space left on device") for no reason --
# `gcloud storage cp` on gs:// paths does this as a metadata-only operation on GCS's
# side, touching zero local disk.
BED_NAME=$(basename "$BED_PREFIX_GS")
for ext in bed bim fam; do
  src="${BED_PREFIX_GS}.${ext}"
  dest="${GRM_INPUT_DIR_GS}/${BED_NAME}.${ext}"
  gcloud storage cp --no-clobber "$src" "$dest"
done

freq_src="${BED_PREFIX_GS}_freq.frq"
freq_dest="${GRM_INPUT_DIR_GS}/${BED_NAME}_freq.frq"

# the .frq file is only ever computed in local scratch (grm_shard_timing.ipynb /
# grm_shard_run.ipynb), never persisted to the bucket on its own -- if it's missing at
# the bucket path, upload it from local scratch first rather than failing here. This
# direction (local -> gs://) is a real upload, but the file is tiny, so it's fine.
if ! gcloud storage ls "$freq_src" >/dev/null 2>&1; then
  local_freq="$HOME/scratch_grm/${BED_NAME}_freq.frq"
  if [ -f "$local_freq" ]; then
    echo "bucket copy of .frq missing -- uploading from local scratch instead"
    gcloud storage cp "$local_freq" "$freq_src"
  else
    echo "no .frq file found at bucket path or local scratch ($local_freq) -- run grm_shard_timing.ipynb's precompute cell first" >&2
    exit 1
  fi
fi

gcloud storage cp --no-clobber "$freq_src" "$freq_dest"

gcloud storage ls -l "$GRM_INPUT_DIR_GS"

## Stage 1 -- troubleshooting job

The smallest possible submission: no real inputs, no plink, just
`echo`/`hostname`/`date` on the cheapest default machine. If this doesn't
come back `SUCCESS` via `dstat` below, the problem is in dsub/Batch/auth
setup, not in anything plink- or GRM-specific -- worth fully resolving
before moving to Stage 2.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
SERVICE_ACCOUNT=$4
NETWORK=$5
SUBNETWORK=$6

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "grm-troubleshoot" \
  --command 'echo "hello from dsub"; hostname; date' \
  > /tmp/troubleshoot_job_id.txt

cat /tmp/troubleshoot_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/troubleshoot_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*'

## Stage 2 -- validation batch (5 real shards)

Only run this once Stage 1 comes back `SUCCESS`. Candidate production
shard count is `N_SHARDS = 300` (per the "fewer, larger shards" direction
from `grm_shard_timing.ipynb` -- adjust once that notebook's
fixed-vs-variable-cost test is back). `VALIDATION_KS` samples 5 shards
spread across the range, same reasoning as that notebook's `SAMPLE_KS`,
rather than committing to all 300 before knowing real per-task cost/timing
on Batch specifically (staging + provisioning overhead per task is not
something the interactive-VM timing captured).

In [ ]:
N_SHARDS = 300   # candidate production count -- adjust per grm_shard_timing.ipynb's fixed-vs-variable-cost test
VALIDATION_KS = [1, 75, 150, 225, 300]   # spread across the full range, not clustered at one end

TASKS_PATH = "/tmp/grm_validation_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env K\t--env N_SHARDS\n")
    for k in VALIDATION_KS:
        f.write(f"{k}\t{N_SHARDS}\n")

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS" "$GRM_INPUT_DIR_GS" "$MACHINE_VCPUS" "$MEMORY_MB" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
GRM_INPUT_DIR_GS=$4
MACHINE_VCPUS=$5
MEMORY_MB=$6
SERVICE_ACCOUNT=$7
NETWORK=$8
SUBNETWORK=$9

# N1 custom machine types need an "-ext" suffix once memory exceeds 8 GiB/vCPU
# (8192 MB/vCPU) -- without it the machine-type string is invalid and Batch rejects
# the VM at provisioning time, before the task (or its log) ever runs. That's almost
# certainly why the last attempt's 5 tasks all failed with zero logs uploaded at all.
MEM_PER_VCPU=$((MEMORY_MB / MACHINE_VCPUS))
if [ "$MEM_PER_VCPU" -gt 8192 ]; then
  MACHINE_TYPE="n1-custom-${MACHINE_VCPUS}-${MEMORY_MB}-ext"
else
  MACHINE_TYPE="n1-custom-${MACHINE_VCPUS}-${MEMORY_MB}"
fi
echo "machine-type: $MACHINE_TYPE ($MEM_PER_VCPU MB/vCPU)"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "grm-shard-validation" \
  --machine-type "$MACHINE_TYPE" \
  --disk-size 100 \
  --input-recursive BED_DIR="$GRM_INPUT_DIR_GS" \
  --output-recursive OUT_DIR="${BUCKET_DIR_GS}/shards/shard_\${K}_of_\${N_SHARDS}" \
  --command '
    set -e
    BIN_DIR=/tmp/bin
    mkdir -p "$BIN_DIR"
    PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
    wget -q -O /tmp/plink.zip "$PLINK_URL"
    unzip -o -q /tmp/plink.zip plink -d "$BIN_DIR"
    chmod +x "$BIN_DIR/plink"

    BED_PREFIX="${BED_DIR}/genome_wide_round2b_thinned_bed"
    FREQ_PATH="${BED_DIR}/genome_wide_round2b_thinned_bed_freq.frq"

    "$BIN_DIR/plink" \
      --bfile "$BED_PREFIX" \
      --make-grm-bin \
      --parallel "${K}" "${N_SHARDS}" \
      --read-freq "$FREQ_PATH" \
      --memory '"$MEMORY_MB"' \
      --threads '"$MACHINE_VCPUS"' \
      --out "${OUT_DIR}/grm_shard_${K}_of_${N_SHARDS}"
  ' \
  --tasks /tmp/grm_validation_tasks.tsv \
  > /tmp/validation_job_id.txt

cat /tmp/validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*'

## Status

**Stage 1 (troubleshooting job) confirmed `SUCCESS`** on 2026-07-17
(16:47:34), via `dstat`. Three real platform issues were found and fixed
along the way, none plink-specific -- exactly what this stage exists to
catch early, before risking anything on a real shard:

1. `PERMISSION_DENIED: ... act as service account ...` -- dsub's default
   (the project's Compute Engine default SA) isn't one the calling
   identity can act as. Fixed by passing `--service-account` with this
   VM's own attached pet SA instead.
2. `no_external_ip_address field is invalid ... VPC Service Controls is
   enabled` -- Batch jobs on this project must run with no external IP,
   on the project's private network/subnet. Fixed by adding
   `--network`/`--subnetwork` (Verily Workbench's fixed `network`/
   `subnetwork` resource names) and `--use-private-address`, per
   Workbench's own dsub guide.
3. `WORKSPACE_BUCKET_GS` was a guess (`gs://<project-id>/...`) that
   pointed at a bucket that doesn't exist (404) -- this, not Private
   Google Access, turned out to be why the job ran but then failed
   (couldn't upload its own log). Fixed by reading the real bucket name
   directly off the gcsfuse mount (`ps -eo pid,args | grep gcsfuse`):
   `shared-env-pilot-wb-fulgent-almond-9474`.

## Next steps

1. Run Stage 2 (validation batch, 5 shards) -- same fixes are wired into
   that `dsub` call. `MACHINE_VCPUS` is now 12 (bumped from an earlier,
   too-conservative guess of 6 -- see the Inputs cell's comment for the
   real timing data behind that), and `MEMORY_MB` is sized from the real
   bed-file-size formula.
2. Compare Stage 2's real per-task wall-clock/cost against the
   interactive `grm_shard_timing.ipynb`/`grm_shard_run.ipynb` numbers --
   Batch tasks have extra overhead (VM provisioning, downloading plink +
   inputs fresh each time) the interactive timing didn't capture, worth
   knowing before extrapolating cost for the full run.
3. Once (1)-(2) look sane and `N_SHARDS` is settled, scale `VALIDATION_KS`
   up to the full `range(1, N_SHARDS + 1)` for the real run -- not built
   as its own cell yet, deliberately, until the above is confirmed.